<a href="https://colab.research.google.com/github/ThiagoOsellame/comprariacomponents.github.io/blob/main/Copia_de_Untitled0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from ipyleaflet import Map, Marker, basemaps, Rectangle
from geopy.geocoders import Nominatim

# 1. Cargar datos base y proyectar 2025
ruta_archivo = '/content/Humedad_Suelo_SMAP_ZonaNucleo_2024.csv'
df = pd.read_csv(ruta_archivo)
df['fecha'] = pd.to_datetime(df['fecha'])

# Clonar los datos de 2024 para simular 2025 en la demostración
df_2025 = df.copy()
df_2025['fecha'] = df_2025['fecha'] + pd.DateOffset(years=1)
df = pd.concat([df, df_2025]).sort_values('fecha').reset_index(drop=True)

# Calcular biomasa (Corregido y validado)
df['ndvi'] = 0.2 + 0.6 * np.exp(-0.2 * ((df['fecha'].dt.month - 10)**2))

# Variable global para el reporte
reporte_texto = ""

# 2. Crear Mapa Interactivo
mapa = Map(center=[-33.0, -61.5], zoom=6, basemap=basemaps.Esri.WorldImagery, layout=widgets.Layout(height='300px', margin='10px 0', border='1px solid #ccc'))
limites_zona_nucleo = ((-35.0, -64.0), (-31.0, -59.0))
recuadro = Rectangle(bounds=limites_zona_nucleo, color='#ff7800', fill_color='#ff7800', fill_opacity=0.2, weight=2)
mapa.add_layer(recuadro)

marcador = Marker(location=[0, 0], draggable=True)
salida_busqueda = widgets.Output()

def actualizar_marcador(**kwargs):
    if kwargs.get('type') == 'click':
        coords = kwargs.get('coordinates')
        marcador.location = coords
        if marcador not in mapa.layers:
            mapa.add_layer(marcador)

        with salida_busqueda:
            clear_output()
            display(widgets.HTML(f"<div style='color: #052c65; background-color: #cfe2ff; padding: 8px; border-radius: 5px; margin-top: 5px;'>📍 <b>Selección Manual en Mapa</b> | Lat: {coords[0]:.4f}, Lon: {coords[1]:.4f}</div>"))

mapa.on_interaction(actualizar_marcador)

# 3. Módulo de Búsqueda por Texto
geolocador = Nominatim(user_agent="space_up_agro")
buscador_texto = widgets.Text(placeholder='Ej: Cruz Alta o 2189', description='Buscar:', style={'description_width': 'initial'}, layout=widgets.Layout(width='70%'))
boton_buscar = widgets.Button(description='Ir', button_style='info', icon='search', layout=widgets.Layout(width='25%'))

def buscar_ubicacion(b):
    with salida_busqueda:
        clear_output()
        texto = buscador_texto.value.strip()
        texto_limpio = texto.replace(" ", "")
# LUEGO SE PUEDE BORRAR EL DICCIONARIO
        diccionario_cp = {
            "2125": "Zavalla, Santa Fe, Argentina",
            "2189": "Cruz Alta, Córdoba, Argentina",
            "2185": "San José de la Esquina, Santa Fe, Argentina",
            "2170": "Casilda, Santa Fe, Argentina",
            "2000": "Rosario, Santa Fe, Argentina"
        }

        ubicacion = None
        if texto_limpio in diccionario_cp:
            ubicacion = geolocador.geocode(diccionario_cp[texto_limpio])
        elif texto_limpio.isdigit():
            ubicacion = geolocador.geocode({'postalcode': texto_limpio, 'country': 'Argentina'})
            if not ubicacion:
                ubicacion = geolocador.geocode(f"{texto_limpio}, Argentina")
        else:
            ubicacion = geolocador.geocode(texto, country_codes='ar')

        if ubicacion:
            marcador.location = (ubicacion.latitude, ubicacion.longitude)
            mapa.center = (ubicacion.latitude, ubicacion.longitude)
            mapa.zoom = 12
            if marcador not in mapa.layers:
                mapa.add_layer(marcador)

            partes_direccion = ubicacion.address.split(',')
            nombre_lugar = partes_direccion[0].strip()
            if nombre_lugar.isdigit() and len(partes_direccion) > 1:
                nombre_lugar = partes_direccion[1].strip()

            display(widgets.HTML(f"<div style='color: #0f5132; background-color: #d1e7dd; padding: 8px; border-radius: 5px; margin-top: 5px;'>📍 <b>{nombre_lugar}</b> | Lat: {ubicacion.latitude:.4f}, Lon: {ubicacion.longitude:.4f}</div>"))
        else:
            display(widgets.HTML("<div style='color: #842029; background-color: #f8d7da; padding: 8px; border-radius: 5px; margin-top: 5px;'>⚠️ No encontrado. OpenStreetMap no tiene este registro, probá ingresar el nombre completo del pueblo.</div>"))

boton_buscar.on_click(buscar_ubicacion)
caja_busqueda = widgets.HBox([buscador_texto, boton_buscar], layout=widgets.Layout(justify_content='space-between'))

# 4. Controles ampliados
selector_cultivo = widgets.Dropdown(
    options=[
        ('Trigo (Jun - Dic)', ('2024-06-15', '2024-12-10')),
        ('Cebada (Jun - Dic)', ('2024-06-01', '2024-12-05')),
        ('Maíz Temprano (Sep - Abr)', ('2024-09-15', '2025-04-15')),
        ('Girasol (Oct - Mar)', ('2024-10-15', '2025-03-15')),
        ('Soja de 1ra (Nov - May)', ('2024-11-01', '2025-05-15')),
        ('Soja de 2da (Dic - May)', ('2024-12-15', '2025-05-20')),
        ('Maíz Tardío (Dic - Jul)', ('2024-12-10', '2025-07-15')),
        ('Sorgo (Nov - Abr)', ('2024-11-15', '2025-04-30')),
        ('Maní (Nov - Abr)', ('2024-11-01', '2025-04-15')),
        ('Cultivo de Servicio / Vicia (Mar - Sep)', ('2024-03-15', '2024-09-15'))
    ],
    description='Cultivo:', style={'description_width': 'initial'}, layout=widgets.Layout(width='100%')
)

slider_umbral = widgets.FloatSlider(value=0.20, min=0.10, max=0.30, step=0.01, description='Buffer (cm³/cm³):', style={'description_width': 'initial'}, layout=widgets.Layout(width='100%'))
boton_analizar = widgets.Button(description='Ejecutar Simulación', button_style='success', icon='rocket', layout=widgets.Layout(width='100%', height='45px'))
boton_exportar = widgets.Button(description='Descargar Reporte', button_style='primary', icon='download', layout=widgets.Layout(width='100%', height='45px'))

salida = widgets.Output(layout=widgets.Layout(min_height='600px', margin='20px 0 0 0'))

# 5. Lógica de Análisis y Exportación (Con semáforo de 3 estados)
def analizar_espacial(b):
    global reporte_texto
    with salida:
        clear_output(wait=True)

        if marcador not in mapa.layers:
            display(widgets.HTML("<div style='color: #842029; background-color: #f8d7da; padding: 15px; border-radius: 8px;'>⚠️ <b>ALTO:</b> Hacé clic en el mapa o buscá una ubicación antes de ejecutar el análisis.</div>"))
            return

        lat, lon = marcador.location
        inicio, fin = pd.to_datetime(selector_cultivo.value[0]), pd.to_datetime(selector_cultivo.value[1])
        umbral = slider_umbral.value

        ciclo = df[(df['fecha'] >= inicio) & (df['fecha'] <= fin)]

        if ciclo.empty:
            display(widgets.HTML("<div style='color: #842029; background-color: #f8d7da; padding: 15px; border-radius: 8px;'>⚠️ <b>AVISO:</b> El archivo CSV no contiene mediciones para las fechas de siembra seleccionadas.</div>"))
            return

        estres = (len(ciclo[ciclo['humedad_superficial'] < umbral]) / len(ciclo)) * 100

        # Lógica de Diagnóstico - Semáforo de 3 niveles
        if estres <= 15:
            estado_titulo = "ÓPTIMO: Capacidad amortiguadora intacta"
            estado_txt = "✅ ÓPTIMO: Capacidad amortiguadora (buffer) intacta."
            parametros = f"Estrés hídrico: {estres:.1f}% (Límite tolerado: {umbral} cm³/cm³). Dinámica de fluidos y pH estabilizados."
            accion = "Continuar con rotación estándar. Humedad suficiente para sostener enlaces orgánicos estructurales."
            ahorro = "No se requieren inversiones correctivas adicionales."
            bg_col, bord_col, text_col, icon, eco_col = "#f3f9f5", "#d1e7dd", "#0f5132", "✅", "#198754"
        elif estres <= 30:
            estado_titulo = "PRECAUCIÓN: Estrés hídrico moderado"
            estado_txt = "⚠️ PRECAUCIÓN: Alerta temprana de degradación."
            parametros = f"Estrés hídrico: {estres:.1f}%. Fluctuaciones de pH detectadas. Los enlaces amida se encuentran bajo tensión térmica."
            accion = "Monitorear de cerca. Evaluar inclusión temprana de cultivo de cobertura si la tendencia persiste."
            ahorro = "Costo preventivo en evaluación (Vicia: 60 USD/ha vs Remediación: 150 USD/ha)."
            bg_col, bord_col, text_col, icon, eco_col = "#fff8e6", "#ffecb5", "#664d03", "⚠️", "#997404"
        else:
            estado_titulo = "CRÍTICO: Quiebre de capacidad buffer edáfica"
            estado_txt = "❌ CRÍTICO: Quiebre de capacidad buffer edáfica."
            parametros = f"Estrés hídrico: {estres:.1f}% de mediciones bajo el límite de {umbral} cm³/cm³. Concentración excesiva de solutos."
            accion = "Interrumpir rotación. Sembrar cultivo de servicio (ej. Vicia) para regenerar nitrógeno y restaurar el pH."
            ahorro = "90 USD/ha (Evita remediación química de 150 USD vs siembra de 60 USD)."
            bg_col, bord_col, text_col, icon, eco_col = "#fff4f4", "#f8d7da", "#842029", "🚨", "#842029"

        reporte_texto = f"--- REPORTE SPACE UP - NASA SMAP ---\nUbicación: Lat {lat:.4f}, Lon {lon:.4f}\nCultivo Seleccionado: {selector_cultivo.label}\n\nRESULTADOS:\n- Estado: {estado_txt}\n- Parámetros: {parametros}\n- Acción Recomendada: {accion}\n- Impacto Económico: {ahorro}\n"

        tarjeta_html = f"""
        <div style="font-family: 'Segoe UI', Roboto, sans-serif; padding: 20px; background-color: {bg_col}; border: 1px solid {bord_col}; border-radius: 8px; color: {text_col}; margin-bottom: 20px; box-shadow: 0 4px 6px rgba(0,0,0,0.05);">
            <h3 style="margin: 0 0 12px 0; display: flex; align-items: center; font-size: 20px;">
                <span style="margin-right: 10px; font-size: 26px;">{icon}</span> {estado_titulo}
            </h3>
            <p style="margin: 0 0 8px 0; font-size: 14px;"><strong>📍 Lote Analizado:</strong> Lat {lat:.4f}, Lon {lon:.4f}</p>
            <p style="margin: 0 0 8px 0; font-size: 14px;"><strong>📊 Parámetros:</strong> {parametros}</p>
            <p style="margin: 0 0 15px 0; font-size: 14px;"><strong>🚜 Acción Sugerida:</strong> {accion}</p>
            <div style="padding-top: 15px; border-top: 1px solid {bord_col};">
                <span style="font-size: 15px; font-weight: bold; color: {eco_col};">💰 Impacto Económico Proyectado:</span>
                <span style="font-size: 15px; color: #333;">{ahorro}</span>
            </div>
        </div>
        """
        display(widgets.HTML(tarjeta_html))

        fig = make_subplots(specs=[[{"secondary_y": True}]])
        fig.add_trace(go.Scatter(x=df['fecha'], y=df['humedad_superficial'], name='Humedad Anual', line=dict(color='lightgray', width=1)), secondary_y=False)
        fig.add_trace(go.Scatter(x=ciclo['fecha'], y=ciclo['humedad_superficial'], name='Humedad Ciclo', mode='lines+markers', line=dict(width=3), marker=dict(color='#1f77b4', size=8)), secondary_y=False)
        fig.add_trace(go.Scatter(x=ciclo['fecha'], y=ciclo['ndvi'], name='NDVI (Biomasa)', line=dict(color='green', dash='dot', width=3)), secondary_y=True)
        fig.add_hline(y=umbral, line_dash="dash", line_color="red", annotation_text="Límite Buffer")

        fig.update_layout(
            template="plotly_white",
            title_text=f"<b>Monitoreo Dinámico:</b> {selector_cultivo.label}",
            height=550,
            autosize=True,
            hovermode="x unified",
            margin=dict(l=20, r=20, t=50, b=20),
            legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
        )
        inicio_zoom = inicio - pd.Timedelta(days=15)
        fin_zoom = fin + pd.Timedelta(days=15)
        fig.update_xaxes(range=[inicio_zoom, fin_zoom], showgrid=True, gridcolor='#f0f0f0')
        fig.update_yaxes(title_text="Humedad (cm³/cm³)", secondary_y=False, range=[0, df['humedad_superficial'].max() + 0.05], showgrid=True, gridcolor='#f0f0f0')
        fig.update_yaxes(title_text="Índice Verde (NDVI)", secondary_y=True, range=[0, 1], showgrid=False)

        display(fig)

def exportar_datos(b):
    if reporte_texto != "":
        with open('Reporte_Agro_SpaceUp.txt', 'w') as f:
            f.write(reporte_texto)
        files.download('Reporte_Agro_SpaceUp.txt')
    else:
        with salida:
            display(widgets.HTML("<div style='color: #842029; background-color: #f8d7da; padding: 15px; border-radius: 8px;'>⚠️ <b>Error:</b> Ejecutá una simulación primero para generar el reporte.</div>"))

boton_analizar.on_click(analizar_espacial)
boton_exportar.on_click(exportar_datos)

# 6. Interfaz UI/UX Mejorada
encabezado = widgets.HTML(
    value="""
    <div style="background-color: #0b192c; padding: 20px 25px; border-radius: 8px; box-shadow: 0 4px 10px rgba(0,0,0,0.15); display: flex; align-items: center;">
        <div>
            <h2 style="color: white; margin: 0; font-family: 'Segoe UI', Roboto, sans-serif; letter-spacing: 0.5px;">🛰️ Space Up Analytics</h2>
            <p style="color: #a0b2c6; margin: 4px 0 0 0; font-family: 'Segoe UI', Roboto, sans-serif; font-size: 14px; text-transform: uppercase; letter-spacing: 1px;">Prototipo de Soporte de Decisiones - NASA Space Apps</p>
        </div>
    </div>
    """
)

panel_izquierdo = widgets.VBox([
    widgets.HTML("<h4 style='margin: 0 0 15px 0; font-family: \"Segoe UI\", Roboto, sans-serif; color: #2c3e50; border-bottom: 2px solid #eee; padding-bottom: 8px;'>📍 1. Ubicación del Lote</h4>"),
    caja_busqueda,
    salida_busqueda,
    mapa
], layout=widgets.Layout(width='60%', padding='20px', border='1px solid #e0e0e0', border_radius='8px', background_color='white', margin='0 15px 0 0', box_shadow='0 2px 4px rgba(0,0,0,0.05)'))

panel_derecho = widgets.VBox([
    widgets.HTML("<h4 style='margin: 0 0 15px 0; font-family: \"Segoe UI\", Roboto, sans-serif; color: #2c3e50; border-bottom: 2px solid #eee; padding-bottom: 8px;'>⚙️ 2. Parámetros Agronómicos</h4>"),
    selector_cultivo,
    slider_umbral,
    widgets.HTML("<hr style='border: 0; border-top: 1px solid white; margin: 25px 0;'>"),
    boton_analizar,
    widgets.HTML("<div style='height: 12px;'></div>"),
    boton_exportar
], layout=widgets.Layout(width='40%', padding='20px', border='1px solid #e0e0e0', border_radius='8px', background_color='white', justify_content='flex-start', box_shadow='0 2px 4px rgba(0,0,0,0.05)'))

cuerpo_principal = widgets.HBox([panel_izquierdo, panel_derecho], layout=widgets.Layout(margin='20px 0 0 0'))
fondo_dashboard = widgets.VBox([encabezado, cuerpo_principal, salida], layout=widgets.Layout(padding='25px', background_color='#f4f7f6'))
display(fondo_dashboard)